In [ ]:
# -*- coding: utf-8 -*-
"""
GUI для управления координатным столом с ожиданием ответа "0" от контроллера
после каждой команды. Исправлена проблема дублирования команд при больших
перемещениях: worker больше не повторяет отправку команды по умолчанию (MOVE_RETRIES=1)
и таймаут ожидания подтверждения увеличен, чтобы контроллер успевал выполнить длительные ходы.
Код готов к копирования и запуску как отдельный файл.

Добавлен планировщик траекторий по относительным движениям (отдельный список),
GUI для управления траекторией имеет те же кнопки, что и Scheduler:
    - Add, Remove selected, Clear, Move Up, Move Down, Next(execute & pop)
    - Run, Stop, Delay, Loop

Исправлено поведение кнопок Run/Stop: теперь Run действительно запускает выполнение
для Trajectory и Scheduler, а Stop корректно останавливает выполнение.

Добавлено: счётчики (Repeat count) в GUI для Scheduler и Trajectory.
Если 'Loop' включён — выполняется бесконечный цикл. Если 'Loop' выключен,
можно ввести целое положительное число повторов N — планировщик выполнит
полную последовательность ровно N раз.

Модификации (по задаче пользователя):
1) Логирование (печать сигнала завершения команды в терминал) выполняется
   ТОЛЬКО для команд, отправленных как часть выполнения шагов Trajectory и
   только для тех шагов траектории, для которых установлен флаг "emit".
2) В интерфейсе Trajectory добавлена возможность ставить/снимать галочку
   "Emit to terminal" для каждого элемента траектории. Состояние сохраняется
   и в JSON (ключ "emit").
"""

import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import serial
import time
import json
import os
import threading
import queue

# ----------------------------
# Настройки порта и файлы
# ----------------------------
PORT = "COM7"
BAUD_RATE = 115200
SERIAL_TIMEOUT = 1.0

POSITIONS_FILE = "positions.json"
SCHEDULE_FILE = "schedule.json"
TRAJECTORY_FILE = "trajectory.json"  # файл для относительных траекторий

# ----------------------------
# Границы (контроллерные и пользовательские)
# ----------------------------
CONTROLLER_LIMITS = {
    1: {"min": -80.0,  "max": 0.0},   # Мотор 1 = Y (контроллер)
    2: {"min": -170.0, "max": 0.0}    # Мотор 2 = X (контроллер)
}

USER_LIMITS = {
    1: {"min": 0.0, "max": abs(CONTROLLER_LIMITS[1]["min"])},   # Y: 0 .. 80
    2: {"min": 0.0, "max": abs(CONTROLLER_LIMITS[2]["min"])}    # X: 0 .. 170
}

DEFAULT_USER_POSITIONS = {
    1: 0.0,
    2: 0.0
}

# ----------------------------
# Глобальные объекты и параметры
# ----------------------------
ser = None
ser_lock = threading.Lock()
positions = {}
serial_available = False

# Scheduler (absolute positions)
schedule_list = []
schedule_lock = threading.Lock()
scheduler_thread = None
scheduler_stop_event = threading.Event()
scheduler_running = False

# Trajectory (relative DX,DY,emit_flag)
# Каждый элемент: (dx: float, dy: float, emit: bool)
trajectory_list = []
trajectory_lock = threading.Lock()
traj_thread = None
traj_stop_event = threading.Event()
traj_running = False

cmd_queue = queue.Queue()        # элементы: (raw_command_str, event, retries, emit_flag)
cmd_worker_thread = None
cmd_worker_stop_event = threading.Event()

# Параметры отправки
MOVE_RESPONSE_TIMEOUT = 10.0
MOVE_RETRIES = 1
INTER_COMMAND_SLEEP = 0.3

# GUI canvas size
CANVAS_W = 420
CANVAS_H = 220
MARGIN = 20

# GUI widgets (будут инициализированы в build_gui)
root = None
canvas = None
x_var = None
y_var = None
dx_var = None
dy_var = None
pos_x_label = None
pos_y_label = None
gui_log_widget = None
schedule_listbox = None
trajectory_listbox = None
btn_connect = None
btn_traj_toggle_emit = None

# Scheduler widgets
sched_x_var = None
sched_y_var = None
sched_delay_var = None
sched_loop_var = None
sched_repeats_var = None
btn_sched_run = None
btn_sched_stop = None

# Trajectory widgets
traj_dx_var = None
traj_dy_var = None
traj_delay_var = None
traj_loop_var = None
traj_repeats_var = None
btn_traj_run = None
btn_traj_stop = None

# ----------------------------
# Утилиты
# ----------------------------
def controller_to_user(motor, c_val):
    return -float(c_val)

def user_to_controller(motor, u_val):
    return -float(u_val)

def clamp_user_value(motor, u_val):
    lim = USER_LIMITS[motor]
    if u_val < lim["min"]:
        return lim["min"]
    if u_val > lim["max"]:
        return lim["max"]
    return u_val

def clamp_controller_value(motor, c_val):
    lim = CONTROLLER_LIMITS[motor]
    if c_val < lim["min"]:
        return lim["min"]
    if c_val > lim["max"]:
        return lim["max"]
    return c_val

def format_mm(mm):
    return f"{mm:.3f}"

# ----------------------------
# Функция вывода сигналов в терминал при окончании команд
# ----------------------------
def emit_command_completion_signal(raw_cmd, success):
    """
    Вывод сигнала в терминал при окончании обработки команды.
    raw_cmd: исходная команда (строка, часто с переводом строки)
    success: boolean — True если получено подтверждение '0' (успех), False иначе
    """
    try:
        status = "OK" if success else "FAIL"
        # Печатать краткое сообщение в stdout (терминал)
        try:
            print(f"[SIGNAL] Command finished: {raw_cmd.strip()} -> {status}")
        except Exception:
            # если печать по какой-то причине не удалась, пробуем safe_log
            try:
                safe_log(f"[SIGNAL] Command finished: {raw_cmd.strip()} -> {status}")
            except Exception:
                pass
        # Попытаться послать звуковой сигнал (ASCII bell). В зависимости от терминала звук может не сработать.
        try:
            print('\a', end='', flush=True)
        except Exception:
            pass
    except Exception:
        pass

# ----------------------------
# Загрузка / сохранение позиций
# ----------------------------
def load_positions_from_file():
    if not os.path.isfile(POSITIONS_FILE):
        return dict(DEFAULT_USER_POSITIONS)
    try:
        with open(POSITIONS_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        return dict(DEFAULT_USER_POSITIONS)
    loaded = {}
    for k, v in data.items():
        try:
            mk = int(k); mv = float(v); loaded[mk] = mv
        except Exception:
            pass
    is_user_format = True
    for m in (1, 2):
        if m not in loaded:
            is_user_format = False; break
        val = loaded[m]; lim = USER_LIMITS[m]
        if not (lim["min"] - 1e-6 <= val <= lim["max"] + 1e-6):
            is_user_format = False; break
    if is_user_format:
        for m in (1, 2):
            if m not in loaded:
                loaded[m] = DEFAULT_USER_POSITIONS[m]
        return loaded
    is_controller_format = True
    for m in (1, 2):
        if m not in loaded:
            is_controller_format = False; break
        val = loaded[m]; lim = CONTROLLER_LIMITS[m]
        if not (lim["min"] - 1e-6 <= val <= lim["max"] + 1e-6):
            is_controller_format = False; break
    if is_controller_format:
        converted = {}
        for m in (1, 2):
            converted[m] = controller_to_user(m, loaded[m])
        return converted
    return dict(DEFAULT_USER_POSITIONS)

def save_positions_to_file():
    tmp = POSITIONS_FILE + ".tmp"
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump({str(k): float(v) for k, v in positions.items()}, f, ensure_ascii=False, indent=2)
        os.replace(tmp, POSITIONS_FILE)
    except Exception:
        pass

# ----------------------------
# Serial: подключение/закрытие
# ----------------------------
def connect_serial(port=PORT, baud_rate=BAUD_RATE, timeout=SERIAL_TIMEOUT):
    global ser, serial_available
    try:
        ser = serial.Serial(port, baud_rate, timeout=timeout)
        time.sleep(1.0)
        serial_available = True
        safe_log(f"Подключено к {port} @ {baud_rate}")
        return True
    except Exception as e:
        ser = None
        serial_available = False
        safe_log(f"Не удалось подключиться к {port}: {e}. Работа в эмуляторе.")
        return False

def close_serial():
    global ser, serial_available
    try:
        if ser is not None and getattr(ser, "is_open", False):
            ser.close()
    except Exception:
        pass
    ser = None
    serial_available = False

# ----------------------------
# Логирование (GUI-потокобезопасно)
# ----------------------------
def _append_log_to_widget(text):
    try:
        gui_log_widget.configure(state="normal")
        gui_log_widget.insert("end", text + "\n")
        gui_log_widget.see("end")
        gui_log_widget.configure(state="disabled")
    except Exception:
        pass

def safe_log(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    if root is None or gui_log_widget is None:
        return
    if threading.current_thread() is threading.main_thread():
        _append_log_to_widget(line)
    else:
        try:
            root.after(0, _append_log_to_widget, line)
        except Exception:
            pass

# ----------------------------
# Worker: отправка команды и ожидание '0'
# ----------------------------
def cmd_worker():
    while not cmd_worker_stop_event.is_set():
        try:
            item = cmd_queue.get(timeout=0.1)
        except queue.Empty:
            continue

        # Поддержка двух форматов для устойчивости (ранние вызовы могли класть 3-элементный кортеж)
        raw_cmd = None; ev = None; retries = None; emit_flag = False
        try:
            if isinstance(item, tuple) and len(item) == 4:
                raw_cmd, ev, retries, emit_flag = item
            elif isinstance(item, tuple) and len(item) == 3:
                raw_cmd, ev, retries = item
                emit_flag = False
            else:
                # неожиданный формат
                cmd_queue.task_done()
                continue
        except Exception:
            cmd_queue.task_done()
            continue

        success = False
        if not serial_available or ser is None or not getattr(ser, "is_open", False):
            success = True
            try:
                ev.set()
            except Exception:
                pass
            # Для соответствия условию: эмитим сигнал в терминал ТОЛЬКО если emit_flag True.
            if emit_flag:
                try:
                    emit_command_completion_signal(raw_cmd, success)
                except Exception:
                    pass
            cmd_queue.task_done()
            time.sleep(INTER_COMMAND_SLEEP)
            continue
        # Отправляем команду ровно один раз (MOVE_RETRIES обычно 1).
        for attempt in range(max(1, retries)):
            try:
                with ser_lock:
                    try:
                        ser.reset_input_buffer()
                    except Exception:
                        pass
                    ser.write(raw_cmd.encode("utf-8"))
                    ser.flush()
            except Exception:
                time.sleep(0.05)
                continue
            start = time.time()
            received = False
            while time.time() - start < MOVE_RESPONSE_TIMEOUT:
                try:
                    line = ser.readline()
                except Exception:
                    break
                if not line:
                    continue
                try:
                    s = line.decode("utf-8", errors="ignore").strip()
                except Exception:
                    try:
                        s = line.decode(errors="ignore").strip()
                    except Exception:
                        s = ""
                if not s:
                    continue
                # Подтверждение принимаем как "0" в строке
                if s == "0" or "0" in s:
                    received = True
                    break
            if received:
                success = True
                break
            # если retries>1 и не получили, сделаем повтор (редко)
            time.sleep(0.05)
        try:
            ev.set()
        except Exception:
            pass
        # ВНИМАНИЕ: печать сигнала в терминал выполняется ТОЛЬКО если emit_flag == True.
        if emit_flag:
            try:
                emit_command_completion_signal(raw_cmd, success)
            except Exception:
                pass
        if not success:
            safe_log(f"Нет подтверждения '0' для команды: {raw_cmd.strip()}")
        cmd_queue.task_done()
        time.sleep(INTER_COMMAND_SLEEP)

def start_cmd_worker():
    global cmd_worker_thread, cmd_worker_stop_event
    if cmd_worker_thread is not None and cmd_worker_thread.is_alive():
        return
    cmd_worker_stop_event.clear()
    cmd_worker_thread = threading.Thread(target=cmd_worker, daemon=True)
    cmd_worker_thread.start()

def stop_cmd_worker():
    global cmd_worker_thread, cmd_worker_stop_event
    cmd_worker_stop_event.set()
    if cmd_worker_thread is not None:
        cmd_worker_thread.join(timeout=1.0)

def enqueue_command_and_wait(raw_command, timeout=MOVE_RESPONSE_TIMEOUT + 1.0, retries=MOVE_RETRIES, emit=False):
    """
    Положить команду в очередь и ждать, пока cmd_worker обработает её.
    Параметр emit (bool) указывает, нужно ли выводить сигнал завершения в терминал
    (логика: мы выводим его ТОЛЬКО для команд, связанных с Trajectory и если пользователь
    для конкретного шага включил галочку).
    """
    ev = threading.Event()
    try:
        cmd_queue.put((raw_command, ev, retries, bool(emit)))
    except Exception:
        return False
    ev.wait(timeout=timeout)
    return ev.is_set()

# ----------------------------
# Отправка перемещений (через очередь)
# ----------------------------
def move_motor_controller_relative(motor, mm, emit=False):
    cmd = f"{motor} {format_mm(mm)}\n"
    return enqueue_command_and_wait(cmd, emit=emit)

# ----------------------------
# GUI-обновления: безопасно из любого потока
# ----------------------------
def _update_gui_positions_impl():
    if root is None:
        return
    x_val = positions.get(2, DEFAULT_USER_POSITIONS[2])
    y_val = positions.get(1, DEFAULT_USER_POSITIONS[1])
    try:
        pos_x_label.configure(text=f"X = {x_val:.3f} мм")
        pos_y_label.configure(text=f"Y = {y_val:.3f} мм")
    except Exception:
        pass
    draw_platform_on_canvas(x_val, y_val)

def update_gui_positions():
    if root is None:
        return
    if threading.current_thread() is threading.main_thread():
        _update_gui_positions_impl()
    else:
        try:
            root.after(0, _update_gui_positions_impl)
        except Exception:
            pass

# ----------------------------
# Рисование сетки и красного креста
# ----------------------------
def draw_platform_on_canvas(x_user, y_user):
    if canvas is None:
        return
    canvas.delete("all")
    max_x = USER_LIMITS[2]["max"]
    max_y = USER_LIMITS[1]["max"]
    left = MARGIN; top = MARGIN; right = CANVAS_W - MARGIN; bottom = CANVAS_H - MARGIN
    canvas.create_rectangle(left, top, right, bottom, outline="#444")
    cols = 10; rows = 10
    for i in range(1, cols):
        x = left + i * (right - left) / cols
        canvas.create_line(x, top, x, bottom, dash=(2, 4), fill="#ddd")
    for j in range(1, rows):
        y = top + j * (bottom - top) / rows
        canvas.create_line(left, y, right, y, dash=(2, 4), fill="#ddd")
    canvas.create_text(left, bottom + 10, anchor="nw", text="X=0", font=("Arial", 8))
    canvas.create_text(right, bottom + 10, anchor="ne", text=f"X={int(max_x)} мм", font=("Arial", 8))
    canvas.create_text(right + 6, top, anchor="nw", text=f"Y={int(max_y)} мм", font=("Arial", 8))
    nx = 0.0 if max_x == 0 else (x_user / max_x)
    ny = 0.0 if max_y == 0 else (y_user / max_y)
    nx = max(0.0, min(1.0, nx)); ny = max(0.0, min(1.0, ny))
    px = left + nx * (right - left)
    py = bottom - ny * (bottom - top)
    cross_size = 8
    canvas.create_line(px - cross_size, py - cross_size, px + cross_size, py + cross_size, fill="red", width=2)
    canvas.create_line(px - cross_size, py + cross_size, px + cross_size, py - cross_size, fill="red", width=2)
    canvas.create_text(left + 6, top + 6, anchor="nw", text=f"X={x_user:.3f} мм", font=("Arial", 9))
    canvas.create_text(left + 6, top + 22, anchor="nw", text=f"Y={y_user:.3f} мм", font=("Arial", 9))

# ----------------------------
# Перемещения: абсолютные/относительные
# ----------------------------
def move_to_absolute_user(target_x_user, target_y_user):
    motor_x = 2; motor_y = 1
    try:
        x_req = float(target_x_user); y_req = float(target_y_user)
    except Exception as e:
        raise ValueError(f"Некорректные входные данные: {e}")
    x_clamped = clamp_user_value(motor_x, x_req); y_clamped = clamp_user_value(motor_y, y_req)
    if x_clamped != x_req or y_clamped != y_req:
        safe_log("Запрошенная координата была обрезана до допустимого диапазона.")
    cur_x = positions.get(motor_x, DEFAULT_USER_POSITIONS[motor_x])
    cur_y = positions.get(motor_y, DEFAULT_USER_POSITIONS[motor_y])
    delta_user_x = x_clamped - cur_x; delta_user_y = y_clamped - cur_y
    eps = 1e-6
    plan = []
    if abs(delta_user_x) > eps:
        delta_ctrl_x = user_to_controller(motor_x, delta_user_x)
        planned_new_x = cur_x + delta_user_x
        cur_ctrl_x = user_to_controller(motor_x, cur_x)
        new_ctrl_abs = cur_ctrl_x + delta_ctrl_x
        new_ctrl_abs_clamped = clamp_controller_value(motor_x, new_ctrl_abs)
        if abs(new_ctrl_abs_clamped - new_ctrl_abs) > 1e-9:
            planned_new_x = controller_to_user(motor_x, new_ctrl_abs_clamped)
            delta_ctrl_x = new_ctrl_abs_clamped - cur_ctrl_x
        plan.append((motor_x, delta_ctrl_x, planned_new_x))
    if abs(delta_user_y) > eps:
        delta_ctrl_y = user_to_controller(motor_y, delta_user_y)
        planned_new_y = cur_y + delta_user_y
        cur_ctrl_y = user_to_controller(motor_y, cur_y)
        new_ctrl_abs = cur_ctrl_y + delta_ctrl_y
        new_ctrl_abs_clamped = clamp_controller_value(motor_y, new_ctrl_abs)
        if abs(new_ctrl_abs_clamped - new_ctrl_abs) > 1e-9:
            planned_new_y = controller_to_user(motor_y, new_ctrl_abs_clamped)
            delta_ctrl_y = new_ctrl_abs_clamped - cur_ctrl_y
        plan.append((motor_y, delta_ctrl_y, planned_new_y))
    if not plan:
        safe_log("Платформа уже в запрошенной позиции или изменения пренебрежимо малы.")
        return
    for motor, delta_ctrl, planned_new in plan:
        # Для абсолютных перемещений emit=False по умолчанию
        ok = move_motor_controller_relative(motor, delta_ctrl, emit=False)
        if not ok:
            safe_log(f"Шаг прерван: motor={motor}, delta={delta_ctrl:.3f} (нет подтверждения)")
            return
        positions[motor] = float(planned_new)
        save_positions_to_file()
        update_gui_positions()
    safe_log(f"Движение завершено. Текущие позиции (user): X={positions[2]:.3f} мм, Y={positions[1]:.3f} мм")

def move_relative_user(dx_user, dy_user, emit=False):
    """
    Выполнение относительных смещений. Параметр emit передаётся в move_motor_controller_relative
    и определяет, будет ли выводиться сигнал завершения в терминал для каждой отправленной
    low-level команды (motor moves).
    """
    motor_x = 2; motor_y = 1
    try:
        dx = float(dx_user); dy = float(dy_user)
    except Exception as e:
        raise ValueError(f"Некорректные относительные смещения: {e}")
    new_x = positions.get(motor_x, DEFAULT_USER_POSITIONS[motor_x]) + dx
    new_y = positions.get(motor_y, DEFAULT_USER_POSITIONS[motor_y]) + dy
    new_x_clamped = clamp_user_value(motor_x, new_x); new_y_clamped = clamp_user_value(motor_y, new_y)
    if new_x_clamped != new_x or new_y_clamped != new_y:
        safe_log("Относительное смещение было скорректировано, чтобы не выйти за границы.")
        dx = new_x_clamped - positions.get(motor_x, DEFAULT_USER_POSITIONS[motor_x])
        dy = new_y_clamped - positions.get(motor_y, DEFAULT_USER_POSITIONS[motor_y])
    if abs(dx) > 1e-6:
        ok = move_motor_controller_relative(motor_x, user_to_controller(motor_x, dx), emit=bool(emit))
        if not ok:
            safe_log("Нет подтверждения при смещении X; операция прервана.")
            return False
        positions[motor_x] = positions.get(motor_x, DEFAULT_USER_POSITIONS[motor_x]) + dx
        save_positions_to_file()
        update_gui_positions()
        safe_log(f"X смещён на {dx:.3f} мм -> X={positions[motor_x]:.3f} мм")
    if abs(dy) > 1e-6:
        ok = move_motor_controller_relative(motor_y, user_to_controller(motor_y, dy), emit=bool(emit))
        if not ok:
            safe_log("Нет подтверждения при смещении Y; операция прервана.")
            return False
        positions[motor_y] = positions.get(motor_y, DEFAULT_USER_POSITIONS[motor_y]) + dy
        save_positions_to_file()
        update_gui_positions()
        safe_log(f"Y смещён на {dy:.3f} мм -> Y={positions[motor_y]:.3f} мм")
    return True

def do_home_procedure():
    safe_log("Запуск процедуры Home.")
    ok = move_motor_controller_relative(1, 200, emit=False)
    if not ok:
        safe_log("Home: нет подтверждения для motor 1.")
        return
    ok = move_motor_controller_relative(2, 200, emit=False)
    if not ok:
        safe_log("Home: нет подтверждения для motor 2.")
        return
    positions[1] = 0.0; positions[2] = 0.0
    save_positions_to_file()
    update_gui_positions()
    safe_log("Home выполнен. Позиции установлены в 0 (user).")

# ----------------------------
# Scheduler: управление очередью и выполнение (абсолютные позиции)
# ----------------------------
def add_to_schedule(x_user, y_user, insert_at_end=True):
    global schedule_list
    try:
        x = float(x_user); y = float(y_user)
    except Exception:
        raise ValueError("Некорректные координаты для добавления в расписание.")
    x = clamp_user_value(2, x); y = clamp_user_value(1, y)
    with schedule_lock:
        if insert_at_end:
            schedule_list.append((x, y))
        else:
            schedule_list.insert(0, (x, y))
    _refresh_schedule_listbox()
    safe_log(f"Добавлено в расписание: X={x:.3f} мм, Y={y:.3f} мм")

def _refresh_schedule_listbox():
    try:
        schedule_listbox.delete(0, "end")
        with schedule_lock:
            for x, y in schedule_list:
                schedule_listbox.insert("end", f"X={x:.3f}  Y={y:.3f}")
    except Exception:
        pass

def remove_selected_schedule():
    sel = schedule_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    with schedule_lock:
        try:
            removed = schedule_list.pop(idx)
        except Exception:
            return
    _refresh_schedule_listbox()
    safe_log(f"Удалено из расписания: X={removed[0]:.3f} мм, Y={removed[1]:.3f} мм")

def clear_schedule():
    global schedule_list
    with schedule_lock:
        schedule_list = []
    _refresh_schedule_listbox()
    safe_log("Расписание очищено.")

def move_schedule_up():
    sel = schedule_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    if idx == 0:
        return
    with schedule_lock:
        schedule_list[idx-1], schedule_list[idx] = schedule_list[idx], schedule_list[idx-1]
    _refresh_schedule_listbox()
    try:
        schedule_listbox.select_set(idx-1)
    except Exception:
        pass

def move_schedule_down():
    sel = schedule_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    with schedule_lock:
        if idx >= len(schedule_list)-1:
            return
        schedule_list[idx+1], schedule_list[idx] = schedule_list[idx], schedule_list[idx+1]
    _refresh_schedule_listbox()
    try:
        schedule_listbox.select_set(idx+1)
    except Exception:
        pass

def save_schedule_to_file(path=SCHEDULE_FILE):
    with schedule_lock:
        try:
            with open(path, "w", encoding="utf-8") as f:
                json.dump([{"x": float(x), "y": float(y)} for x, y in schedule_list], f, ensure_ascii=False, indent=2)
            safe_log(f"Расписание сохранено в {path}")
        except Exception as e:
            safe_log(f"Ошибка при сохранении расписания: {e}")

def load_schedule_from_file(path=SCHEDULE_FILE):
    global schedule_list
    if not os.path.isfile(path):
        safe_log(f"Файл расписания {path} не найден.")
        return
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        safe_log(f"Не удалось прочитать {path}: {e}")
        return
    new_list = []
    for item in data:
        try:
            x = float(item.get("x", 0.0)); y = float(item.get("y", 0.0))
            x = clamp_user_value(2, x); y = clamp_user_value(1, y)
            new_list.append((x, y))
        except Exception:
            pass
    with schedule_lock:
        schedule_list = new_list
    _refresh_schedule_listbox()
    safe_log(f"Расписание загружено из {path} ({len(schedule_list)} пунктов).")

def _set_sched_buttons_running(running):
    # Обновляем состояния кнопок Scheduler (Run / Stop)
    def _update():
        try:
            if running:
                btn_sched_run.configure(state="disabled")
                btn_sched_stop.configure(state="normal")
            else:
                btn_sched_run.configure(state="normal")
                btn_sched_stop.configure(state="disabled")
        except Exception:
            pass
    if root is not None and threading.current_thread() is threading.main_thread():
        _update()
    elif root is not None:
        try:
            root.after(0, _update)
        except Exception:
            pass
    else:
        _update()

def run_schedule_thread(delay_between=0.5, loop=False, repeats=1):
    """
    Выполняет расписание:
    - Если loop=True — выполняет бесконечно (пока не stop_event)
    - Иначе выполняет последовательность 'repeats' раз (repeats >= 1)
    """
    global scheduler_running
    safe_log("Запуск выполнения расписания...")
    scheduler_stop_event.clear()
    scheduler_running = True
    _set_sched_buttons_running(True)
    try:
        if loop:
            # бесконечный цикл
            while not scheduler_stop_event.is_set():
                with schedule_lock:
                    seq = list(schedule_list)
                if not seq:
                    safe_log("Расписание пустое — прекращаю выполнение.")
                    break
                for idx, (x, y) in enumerate(seq):
                    if scheduler_stop_event.is_set():
                        break
                    safe_log(f"Выполнение шага {idx+1}/{len(seq)}: X={x:.3f} мм, Y={y:.3f} мм")
                    try:
                        move_to_absolute_user(x, y)
                    except Exception as e:
                        safe_log(f"Ошибка при выполнении шага {idx+1}: {e}")
                    t_sleep = 0.0
                    while t_sleep < delay_between and not scheduler_stop_event.is_set():
                        time.sleep(min(0.1, delay_between - t_sleep))
                        t_sleep += 0.1
        else:
            # ограниченное число повторов
            try:
                repeats_int = int(repeats)
            except Exception:
                repeats_int = 1
            if repeats_int < 1:
                repeats_int = 1
            for rep in range(repeats_int):
                if scheduler_stop_event.is_set():
                    break
                safe_log(f"Запуск прохода {rep+1}/{repeats_int} по расписанию.")
                with schedule_lock:
                    seq = list(schedule_list)
                if not seq:
                    safe_log("Расписание пустое — прекращаю выполнение.")
                    break
                for idx, (x, y) in enumerate(seq):
                    if scheduler_stop_event.is_set():
                        break
                    safe_log(f"Выполнение шага {idx+1}/{len(seq)} (проход {rep+1}/{repeats_int}): X={x:.3f} мм, Y={y:.3f} мм")
                    try:
                        move_to_absolute_user(x, y)
                    except Exception as e:
                        safe_log(f"Ошибка при выполнении шага {idx+1}: {e}")
                    t_sleep = 0.0
                    while t_sleep < delay_between and not scheduler_stop_event.is_set():
                        time.sleep(min(0.1, delay_between - t_sleep))
                        t_sleep += 0.1
    finally:
        scheduler_running = False
        _set_sched_buttons_running(False)
        safe_log("Выполнение расписания остановлено.")

def start_run_schedule(delay_between=0.5, loop=False, repeats=1):
    global scheduler_thread
    if scheduler_running:
        safe_log("Расписание уже выполняется.")
        return
    scheduler_stop_event.clear()
    scheduler_thread = threading.Thread(target=run_schedule_thread, args=(delay_between, loop, repeats), daemon=True)
    scheduler_thread.start()

def stop_run_schedule():
    scheduler_stop_event.set()
    safe_log("Запрос на остановку выполнения расписания получен.")

def run_next_in_schedule():
    with schedule_lock:
        if not schedule_list:
            safe_log("Расписание пустое — нечего выполнять.")
            return
        x, y = schedule_list.pop(0)
    _refresh_schedule_listbox()
    safe_log(f"Выполнение следующего шага: X={x:.3f} мм, Y={y:.3f} мм")
    def _t():
        try:
            move_to_absolute_user(x, y)
        except Exception as e:
            safe_log(f"Ошибка при выполнении Next: {e}")
    threading.Thread(target=_t, daemon=True).start()

# ----------------------------
# Trajectory: относительные движения (DX,DY,emit)
# ----------------------------
def add_to_trajectory(dx_user, dy_user, insert_at_end=True, emit=False):
    global trajectory_list
    try:
        dx = float(dx_user); dy = float(dy_user)
    except Exception:
        raise ValueError("Некорректные значения DX/DY для добавления в траекторию.")
    with trajectory_lock:
        if insert_at_end:
            trajectory_list.append((dx, dy, bool(emit)))
        else:
            trajectory_list.insert(0, (dx, dy, bool(emit)))
    _refresh_trajectory_listbox()
    safe_log(f"Добавлено в траекторию: DX={dx:.3f} мм, DY={dy:.3f} мм, emit={bool(emit)}")

def _refresh_trajectory_listbox():
    try:
        trajectory_listbox.delete(0, "end")
        with trajectory_lock:
            for dx, dy, emit in trajectory_list:
                mark = "[x]" if emit else "[ ]"
                trajectory_listbox.insert("end", f"{mark} DX={dx:.3f}  DY={dy:.3f}")
    except Exception:
        pass

def remove_selected_trajectory():
    sel = trajectory_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    with trajectory_lock:
        try:
            removed = trajectory_list.pop(idx)
        except Exception:
            return
    _refresh_trajectory_listbox()
    safe_log(f"Удалено из траектории: DX={removed[0]:.3f} мм, DY={removed[1]:.3f} мм, emit={removed[2]}")

def clear_trajectory():
    global trajectory_list
    with trajectory_lock:
        trajectory_list = []
    _refresh_trajectory_listbox()
    safe_log("Траектория очищена.")

def move_trajectory_up():
    sel = trajectory_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    if idx == 0:
        return
    with trajectory_lock:
        trajectory_list[idx-1], trajectory_list[idx] = trajectory_list[idx], trajectory_list[idx-1]
    _refresh_trajectory_listbox()
    try:
        trajectory_listbox.select_set(idx-1)
    except Exception:
        pass

def move_trajectory_down():
    sel = trajectory_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    with trajectory_lock:
        if idx >= len(trajectory_list)-1:
            return
        trajectory_list[idx+1], trajectory_list[idx] = trajectory_list[idx], trajectory_list[idx+1]
    _refresh_trajectory_listbox()
    try:
        trajectory_listbox.select_set(idx+1)
    except Exception:
        pass

def save_trajectory_to_file(path=TRAJECTORY_FILE):
    with trajectory_lock:
        try:
            with open(path, "w", encoding="utf-8") as f:
                json.dump([{"dx": float(dx), "dy": float(dy), "emit": bool(emit)} for dx, dy, emit in trajectory_list], f, ensure_ascii=False, indent=2)
            safe_log(f"Траектория сохранена в {path}")
        except Exception as e:
            safe_log(f"Ошибка при сохранении траектории: {e}")

def load_trajectory_from_file(path=TRAJECTORY_FILE):
    global trajectory_list
    if not os.path.isfile(path):
        safe_log(f"Файл траектории {path} не найден.")
        return
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        safe_log(f"Не удалось прочитать {path}: {e}")
        return
    new_list = []
    for item in data:
        try:
            dx = float(item.get("dx", 0.0)); dy = float(item.get("dy", 0.0))
            emit = bool(item.get("emit", False))
            new_list.append((dx, dy, emit))
        except Exception:
            pass
    with trajectory_lock:
        trajectory_list = new_list
    _refresh_trajectory_listbox()
    safe_log(f"Траектория загружена из {path} ({len(trajectory_list)} пунктов).")

def toggle_emit_selected_trajectory():
    """
    Переключает флаг emit для выбранного элемента траектории.
    """
    sel = trajectory_listbox.curselection()
    if not sel:
        return
    idx = sel[0]
    with trajectory_lock:
        try:
            dx, dy, emit = trajectory_list[idx]
            trajectory_list[idx] = (dx, dy, not emit)
            new_emit = not emit
        except Exception:
            return
    _refresh_trajectory_listbox()
    try:
        trajectory_listbox.select_set(idx)
    except Exception:
        pass
    safe_log(f"Toggle emit для траектории[{idx}] -> {new_emit}")

def _set_traj_buttons_running(running):
    # Обновляем состояния кнопок Trajectory (Run / Stop)
    def _update():
        try:
            if running:
                btn_traj_run.configure(state="disabled")
                btn_traj_stop.configure(state="normal")
            else:
                btn_traj_run.configure(state="normal")
                btn_traj_stop.configure(state="disabled")
        except Exception:
            pass
    if root is not None and threading.current_thread() is threading.main_thread():
        _update()
    elif root is not None:
        try:
            root.after(0, _update)
        except Exception:
            pass
    else:
        _update()

def run_trajectory_thread(delay_between=0.5, loop=False, repeats=1):
    """
    Выполняет траекторию относительных перемещений:
    - Если loop=True — выполняет бесконечно (пока не stop_event)
    - Иначе выполняет последовательность 'repeats' раз (repeats >= 1)
    При выполнении каждого шага учитывается флаг emit для конкретного пункта —
    если он True, то при окончании каждой low-level команды будет выведен сигнал
    в терминал (см. cmd_worker обработку emit_flag).
    """
    global traj_running
    safe_log("Запуск выполнения траектории (относительные перемещения)...")
    traj_stop_event.clear()
    traj_running = True
    _set_traj_buttons_running(True)
    try:
        if loop:
            while not traj_stop_event.is_set():
                with trajectory_lock:
                    seq = list(trajectory_list)
                if not seq:
                    safe_log("Траектория пустая — прекращаю выполнение.")
                    break
                for idx, item in enumerate(seq):
                    if traj_stop_event.is_set():
                        break
                    dx, dy, emit = item
                    safe_log(f"Выполнение шага траектории {idx+1}/{len(seq)}: DX={dx:.3f} мм, DY={dy:.3f} мм, emit={emit}")
                    try:
                        ok = move_relative_user(dx, dy, emit=emit)
                        if not ok:
                            safe_log(f"Ошибка/прерывание при относительном шаге {idx+1}")
                            traj_stop_event.set()
                            break
                    except Exception as e:
                        safe_log(f"Ошибка при выполнении относительного шага {idx+1}: {e}")
                    t_sleep = 0.0
                    while t_sleep < delay_between and not traj_stop_event.is_set():
                        time.sleep(min(0.1, delay_between - t_sleep))
                        t_sleep += 0.1
        else:
            try:
                repeats_int = int(repeats)
            except Exception:
                repeats_int = 1
            if repeats_int < 1:
                repeats_int = 1
            for rep in range(repeats_int):
                if traj_stop_event.is_set():
                    break
                safe_log(f"Запуск прохода {rep+1}/{repeats_int} по траектории.")
                with trajectory_lock:
                    seq = list(trajectory_list)
                if not seq:
                    safe_log("Траектория пустая — прекращаю выполнение.")
                    break
                for idx, item in enumerate(seq):
                    if traj_stop_event.is_set():
                        break
                    dx, dy, emit = item
                    safe_log(f"Выполнение шага траектории {idx+1}/{len(seq)} (проход {rep+1}/{repeats_int}): DX={dx:.3f} мм, DY={dy:.3f} мм, emit={emit}")
                    try:
                        ok = move_relative_user(dx, dy, emit=emit)
                        if not ok:
                            safe_log(f"Ошибка/прерывание при относительном шаге {idx+1}")
                            traj_stop_event.set()
                            break
                    except Exception as e:
                        safe_log(f"Ошибка при выполнении относительного шага {idx+1}: {e}")
                    t_sleep = 0.0
                    while t_sleep < delay_between and not traj_stop_event.is_set():
                        time.sleep(min(0.1, delay_between - t_sleep))
                        t_sleep += 0.1
    finally:
        traj_running = False
        _set_traj_buttons_running(False)
        safe_log("Выполнение траектории остановлено.")

def start_run_trajectory(delay_between=0.5, loop=False, repeats=1):
    global traj_thread
    if traj_running:
        safe_log("Траектория уже выполняется.")
        return
    traj_stop_event.clear()
    traj_thread = threading.Thread(target=run_trajectory_thread, args=(delay_between, loop, repeats), daemon=True)
    traj_thread.start()

def stop_run_trajectory():
    traj_stop_event.set()
    safe_log("Запрос на остановку выполнения траектории получен.")

def run_next_in_trajectory():
    with trajectory_lock:
        if not trajectory_list:
            safe_log("Траектория пустая — нечего выполнять.")
            return
        dx, dy, emit = trajectory_list.pop(0)
    _refresh_trajectory_listbox()
    safe_log(f"Выполнение следующего шага траектории: DX={dx:.3f} мм, DY={dy:.3f} мм, emit={emit}")
    def _t():
        try:
            ok = move_relative_user(dx, dy, emit=emit)
            if not ok:
                safe_log("Next (trajectory): нет подтверждения при выполнении шага.")
        except Exception as e:
            safe_log(f"Ошибка при выполнении Next (trajectory): {e}")
    threading.Thread(target=_t, daemon=True).start()

# ----------------------------
# GUI callbacks и интерфейс
# ----------------------------
def on_move_absolute():
    try:
        x = float(x_var.get()); y = float(y_var.get())
    except Exception:
        messagebox.showerror("Ошибка ввода", "Введите корректные числовые значения для X и Y.")
        return
    t = threading.Thread(target=_threaded_move_absolute, args=(x, y), daemon=True)
    t.start()

def _threaded_move_absolute(x, y):
    try:
        move_to_absolute_user(x, y)
    except Exception as e:
        safe_log(f"Ошибка при move_to_absolute: {e}")
        try:
            messagebox.showerror("Ошибка", f"Не удалось выполнить перемещение: {e}")
        except Exception:
            pass

def on_move_relative():
    try:
        dx = float(dx_var.get()); dy = float(dy_var.get())
    except Exception:
        messagebox.showerror("Ошибка ввода", "Введите корректные числовые значения для DX и DY.")
        return
    t = threading.Thread(target=_threaded_move_relative, args=(dx, dy), daemon=True)
    t.start()

def _threaded_move_relative(dx, dy):
    try:
        # по умолчанию emit=False для ручного относительного перемещения
        move_relative_user(dx, dy, emit=False)
    except Exception as e:
        safe_log(f"Ошибка при move_relative: {e}")
        try:
            messagebox.showerror("Ошибка", f"Не удалось выполнить относительное перемещение: {e}")
        except Exception:
            pass

def on_home():
    t = threading.Thread(target=_threaded_home, daemon=True)
    t.start()

def _threaded_home():
    try:
        do_home_procedure()
    except Exception as e:
        safe_log(f"Ошибка при Home: {e}")
        try:
            messagebox.showerror("Ошибка", f"Не удалось выполнить Home: {e}")
        except Exception:
            pass

def on_save_positions():
    save_positions_to_file()
    safe_log(f"Позиции сохранены в {POSITIONS_FILE}")

def on_connect_serial():
    try:
        btn_connect.configure(state="disabled")
    except Exception:
        pass
    ok = connect_serial()
    start_cmd_worker()
    if ok:
        try:
            messagebox.showinfo("Serial", f"Успешно подключено к {PORT}")
        except Exception:
            pass
    else:
        try:
            messagebox.showwarning("Serial", f"Не удалось подключиться к {PORT}. Работа в эмуляторе.")
        except Exception:
            pass
    try:
        btn_connect.configure(state="normal")
    except Exception:
        pass

# Scheduler callbacks
def on_sched_add():
    try:
        x = float(sched_x_var.get()); y = float(sched_y_var.get())
    except Exception:
        messagebox.showerror("Ошибка ввода", "Введите корректные числовые значения для X и Y (scheduler).")
        return
    add_to_schedule(x, y)

def on_sched_remove():
    remove_selected_schedule()

def on_sched_clear():
    if messagebox.askyesno("Очистить", "Очистить всё расписание?"):
        clear_schedule()

def on_sched_save():
    path = filedialog.asksaveasfilename(defaultextension=".json", filetypes=[("JSON files","*.json"),("All files","*.*")], initialfile=SCHEDULE_FILE)
    if not path:
        return
    save_schedule_to_file(path)

def on_sched_load():
    path = filedialog.askopenfilename(filetypes=[("JSON files","*.json"),("All files","*.*")], initialfile=SCHEDULE_FILE)
    if not path:
        return
    load_schedule_from_file(path)

def on_sched_run():
    try:
        delay = float(sched_delay_var.get())
        if delay < 0:
            raise ValueError()
    except Exception:
        messagebox.showerror("Ошибка ввода", "Введите корректный неотрицательный интервал задержки (сек).")
        return
    loop = bool(sched_loop_var.get())
    if loop:
        repeats = 1
    else:
        try:
            repeats = int(sched_repeats_var.get())
            if repeats < 1:
                raise ValueError()
        except Exception:
            messagebox.showerror("Ошибка ввода", "Введите корректное целое положительное число повторов (Repeat count).")
            return
    start_run_schedule(delay_between=delay, loop=loop, repeats=repeats)

def on_sched_stop():
    stop_run_schedule()

def on_sched_next():
    run_next_in_schedule()

def on_sched_move_up():
    move_schedule_up()

def on_sched_move_down():
    move_schedule_down()

# Trajectory callbacks
def on_traj_add():
    try:
        dx = float(traj_dx_var.get()); dy = float(traj_dy_var.get())
    except Exception:
        messagebox.showerror("Ошибка ввода", "Введите корректные числовые значения для DX и DY (trajectory).")
        return
    # При добавлении по умолчанию emit=False
    add_to_trajectory(dx, dy, emit=False)

def on_traj_remove():
    remove_selected_trajectory()

def on_traj_clear():
    if messagebox.askyesno("Очистить", "Очистить всю траекторию?"):
        clear_trajectory()

def on_traj_save():
    path = filedialog.asksaveasfilename(defaultextension=".json", filetypes=[("JSON files","*.json"),("All files","*.*")], initialfile=TRAJECTORY_FILE)
    if not path:
        return
    save_trajectory_to_file(path)

def on_traj_load():
    path = filedialog.askopenfilename(filetypes=[("JSON files","*.json"),("All files","*.*")], initialfile=TRAJECTORY_FILE)
    if not path:
        return
    load_trajectory_from_file(path)

def on_traj_run():
    try:
        delay = float(traj_delay_var.get())
        if delay < 0:
            raise ValueError()
    except Exception:
        messagebox.showerror("Ошибка ввода", "Введите корректный неотрицательный интервал задержки (сек) для траектории.")
        return
    loop = bool(traj_loop_var.get())
    if loop:
        repeats = 1
    else:
        try:
            repeats = int(traj_repeats_var.get())
            if repeats < 1:
                raise ValueError()
        except Exception:
            messagebox.showerror("Ошибка ввода", "Введите корректное целое положительное число повторов (Repeat count).")
            return
    start_run_trajectory(delay_between=delay, loop=loop, repeats=repeats)

def on_traj_stop():
    stop_run_trajectory()

def on_traj_next():
    run_next_in_trajectory()

def on_traj_move_up():
    move_trajectory_up()

def on_traj_move_down():
    move_trajectory_down()

def on_traj_toggle_emit():
    toggle_emit_selected_trajectory()

# ----------------------------
# Exit: корректное завершение
# ----------------------------
def on_exit():
    try:
        btn_connect.configure(state="disabled")
    except Exception:
        pass
    stop_run_schedule()
    stop_run_trajectory()
    stop_cmd_worker()
    # Дать небольшое время воркерам для завершения
    time.sleep(0.05)
    close_serial()
    try:
        if root is not None:
            root.destroy()
    except Exception:
        try:
            root.quit()
        except Exception:
            pass

# ----------------------------
# Построение GUI
# ----------------------------
def build_gui():
    global root, canvas, x_var, y_var, dx_var, dy_var
    global pos_x_label, pos_y_label, gui_log_widget, btn_connect, btn_traj_toggle_emit
    global sched_x_var, sched_y_var, sched_delay_var, sched_loop_var, sched_repeats_var
    global schedule_listbox, btn_sched_run, btn_sched_stop
    global trajectory_listbox, traj_dx_var, traj_dy_var, traj_delay_var, traj_loop_var, traj_repeats_var, btn_traj_run, btn_traj_stop

    root = tk.Tk()
    root.title("Управление координатным столом — GUI с Scheduler и Trajectory")
    root.geometry("1100x760")
    root.protocol("WM_DELETE_WINDOW", on_exit)

    mainframe = ttk.Frame(root, padding="8 8 8 8")
    mainframe.pack(fill="both", expand=True)

    left_frame = ttk.Frame(mainframe)
    left_frame.grid(row=0, column=0, sticky="nsew", padx=6, pady=6)
    mainframe.columnconfigure(0, weight=1)
    mainframe.columnconfigure(1, weight=0)
    mainframe.rowconfigure(0, weight=1)

    canvas_frame = ttk.LabelFrame(left_frame, text="Позиция платформы")
    canvas_frame.pack(fill="both", expand=False, padx=4, pady=4)
    global CANVAS_W, CANVAS_H
    canvas = tk.Canvas(canvas_frame, width=CANVAS_W, height=CANVAS_H, bg="#f8f8f8")
    canvas.pack(padx=6, pady=6)

    pos_frame = ttk.Frame(left_frame)
    pos_frame.pack(fill="x", padx=4, pady=2)
    pos_x_label = ttk.Label(pos_frame, text="X = 0.000 мм")
    pos_x_label.pack(side="left", padx=6)
    pos_y_label = ttk.Label(pos_frame, text="Y = 0.000 мм")
    pos_y_label.pack(side="left", padx=6)

    log_frame = ttk.LabelFrame(left_frame, text="Лог и сообщения")
    log_frame.pack(fill="both", expand=True, padx=4, pady=4)
    gui_log_widget = tk.Text(log_frame, height=20, state="disabled", wrap="none")
    gui_log_widget.pack(fill="both", expand=True, padx=4, pady=4)

    ctl_frame = ttk.Frame(mainframe)
    ctl_frame.grid(row=0, column=1, sticky="nsew", padx=6, pady=6)

    abs_frame = ttk.LabelFrame(ctl_frame, text="Абсолютное перемещение (пользовательские мм)")
    abs_frame.pack(fill="x", padx=4, pady=4)

    x_var = tk.StringVar(value=str(positions.get(2, 0.0)))
    y_var = tk.StringVar(value=str(positions.get(1, 0.0)))

    ttk.Label(abs_frame, text=f"X (0..{USER_LIMITS[2]['max']:.0f})").grid(row=0, column=0, sticky="w", padx=4, pady=2)
    ttk.Entry(abs_frame, width=12, textvariable=x_var).grid(row=0, column=1, sticky="w", padx=4, pady=2)
    ttk.Label(abs_frame, text=f"Y (0..{USER_LIMITS[1]['max']:.0f})").grid(row=1, column=0, sticky="w", padx=4, pady=2)
    ttk.Entry(abs_frame, width=12, textvariable=y_var).grid(row=1, column=1, sticky="w", padx=4, pady=2)
    ttk.Button(abs_frame, text="Move Absolute", command=on_move_absolute).grid(row=2, column=0, columnspan=2, pady=6)

    rel_frame = ttk.LabelFrame(ctl_frame, text="Относительное смещение (пользовательские мм)")
    rel_frame.pack(fill="x", padx=4, pady=4)

    dx_var = tk.StringVar(value="0.0")
    dy_var = tk.StringVar(value="0.0")

    ttk.Label(rel_frame, text="DX").grid(row=0, column=0, sticky="w", padx=4, pady=2)
    ttk.Entry(rel_frame, width=10, textvariable=dx_var).grid(row=0, column=1, padx=4, pady=2)
    ttk.Label(rel_frame, text="DY").grid(row=1, column=0, sticky="w", padx=4, pady=2)
    ttk.Entry(rel_frame, width=10, textvariable=dy_var).grid(row=1, column=1, padx=4, pady=2)
    ttk.Button(rel_frame, text="Move Relative", command=on_move_relative).grid(row=2, column=0, columnspan=2, pady=6)

    ops_frame = ttk.Frame(ctl_frame)
    ops_frame.pack(fill="x", padx=4, pady=6)
    ttk.Button(ops_frame, text="Home", command=on_home).grid(row=0, column=0, padx=4)
    ttk.Button(ops_frame, text="Save positions", command=on_save_positions).grid(row=0, column=1, padx=4)
    btn_connect = ttk.Button(ops_frame, text=f"Connect ({PORT})", command=on_connect_serial)
    btn_connect.grid(row=0, column=2, padx=4)

    # Scheduler for absolute positions
    sched_frame = ttk.LabelFrame(ctl_frame, text="Scheduler — планировщик абсолютных позиций")
    sched_frame.pack(fill="both", expand=True, padx=4, pady=4)

    sched_inputs = ttk.Frame(sched_frame)
    sched_inputs.pack(fill="x", padx=4, pady=2)

    sched_x_var = tk.StringVar(value=str(positions.get(2, 0.0)))
    sched_y_var = tk.StringVar(value=str(positions.get(1, 0.0)))
    sched_delay_var = tk.StringVar(value="0.5")
    sched_loop_var = tk.IntVar(value=0)
    sched_repeats_var = tk.StringVar(value="1")

    ttk.Label(sched_inputs, text="X").grid(row=0, column=0, padx=2, pady=2, sticky="w")
    ttk.Entry(sched_inputs, width=10, textvariable=sched_x_var).grid(row=0, column=1, padx=2, pady=2)
    ttk.Label(sched_inputs, text="Y").grid(row=0, column=2, padx=2, pady=2, sticky="w")
    ttk.Entry(sched_inputs, width=10, textvariable=sched_y_var).grid(row=0, column=3, padx=2, pady=2)
    ttk.Button(sched_inputs, text="Add to schedule", command=on_sched_add).grid(row=0, column=4, padx=6)

    lb_frame = ttk.Frame(sched_frame)
    lb_frame.pack(fill="both", expand=True, padx=4, pady=4)
    schedule_listbox = tk.Listbox(lb_frame, height=8)
    schedule_listbox.pack(side="left", fill="both", expand=True)
    lb_scroll = ttk.Scrollbar(lb_frame, orient="vertical", command=schedule_listbox.yview)
    lb_scroll.pack(side="right", fill="y")
    schedule_listbox.configure(yscrollcommand=lb_scroll.set)

    sched_ctrl = ttk.Frame(sched_frame)
    sched_ctrl.pack(fill="x", padx=4, pady=2)
    ttk.Button(sched_ctrl, text="Remove selected", command=on_sched_remove).grid(row=0, column=0, padx=2)
    ttk.Button(sched_ctrl, text="Clear", command=on_sched_clear).grid(row=0, column=1, padx=2)
    ttk.Button(sched_ctrl, text="Move Up", command=on_sched_move_up).grid(row=0, column=2, padx=2)
    ttk.Button(sched_ctrl, text="Move Down", command=on_sched_move_down).grid(row=0, column=3, padx=2)
    ttk.Button(sched_ctrl, text="Next (execute & pop)", command=on_sched_next).grid(row=0, column=4, padx=6)

    run_ctrl = ttk.Frame(sched_frame)
    run_ctrl.pack(fill="x", padx=4, pady=2)
    ttk.Label(run_ctrl, text="Delay (s)").grid(row=0, column=0, padx=2)
    ttk.Entry(run_ctrl, width=6, textvariable=sched_delay_var).grid(row=0, column=1, padx=2)
    ttk.Label(run_ctrl, text="Repeat").grid(row=0, column=2, padx=2)
    ttk.Entry(run_ctrl, width=6, textvariable=sched_repeats_var).grid(row=0, column=3, padx=2)
    ttk.Checkbutton(run_ctrl, text="Loop", variable=sched_loop_var).grid(row=0, column=4, padx=4)

    btn_sched_run = ttk.Button(run_ctrl, text="Run", command=on_sched_run)
    btn_sched_run.grid(row=0, column=5, padx=6)
    btn_sched_stop = ttk.Button(run_ctrl, text="Stop", command=on_sched_stop)
    btn_sched_stop.grid(row=0, column=6, padx=6)
    # Изначально Stop неактивна
    btn_sched_stop.configure(state="disabled")
    ttk.Button(run_ctrl, text="Save schedule...", command=on_sched_save).grid(row=0, column=7, padx=6)
    ttk.Button(run_ctrl, text="Load schedule...", command=on_sched_load).grid(row=0, column=8, padx=6)

    # Trajectory frame (relative moves) — GUI содержит те же элементы управления, что и Scheduler
    traj_frame = ttk.LabelFrame(ctl_frame, text="Trajectory — планировщик относительных смещений (DX,DY)")
    traj_frame.pack(fill="both", expand=True, padx=4, pady=4)

    traj_inputs = ttk.Frame(traj_frame)
    traj_inputs.pack(fill="x", padx=4, pady=2)

    traj_dx_var = tk.StringVar(value="0.0")
    traj_dy_var = tk.StringVar(value="0.0")
    traj_delay_var = tk.StringVar(value="0.5")
    traj_loop_var = tk.IntVar(value=0)
    traj_repeats_var = tk.StringVar(value="1")

    ttk.Label(traj_inputs, text="DX").grid(row=0, column=0, padx=2, pady=2, sticky="w")
    ttk.Entry(traj_inputs, width=8, textvariable=traj_dx_var).grid(row=0, column=1, padx=2, pady=2)
    ttk.Label(traj_inputs, text="DY").grid(row=0, column=2, padx=2, pady=2, sticky="w")
    ttk.Entry(traj_inputs, width=8, textvariable=traj_dy_var).grid(row=0, column=3, padx=2, pady=2)
    ttk.Button(traj_inputs, text="Add to trajectory", command=on_traj_add).grid(row=0, column=4, padx=6)

    lb_frame2 = ttk.Frame(traj_frame)
    lb_frame2.pack(fill="both", expand=True, padx=4, pady=4)
    trajectory_listbox = tk.Listbox(lb_frame2, height=8)
    trajectory_listbox.pack(side="left", fill="both", expand=True)
    lb2_scroll = ttk.Scrollbar(lb_frame2, orient="vertical", command=trajectory_listbox.yview)
    lb2_scroll.pack(side="right", fill="y")
    trajectory_listbox.configure(yscrollcommand=lb2_scroll.set)

    traj_ctrl = ttk.Frame(traj_frame)
    traj_ctrl.pack(fill="x", padx=4, pady=2)
    ttk.Button(traj_ctrl, text="Remove selected", command=on_traj_remove).grid(row=0, column=0, padx=2)
    ttk.Button(traj_ctrl, text="Clear", command=on_traj_clear).grid(row=0, column=1, padx=2)
    ttk.Button(traj_ctrl, text="Move Up", command=on_traj_move_up).grid(row=0, column=2, padx=2)
    ttk.Button(traj_ctrl, text="Move Down", command=on_traj_move_down).grid(row=0, column=3, padx=2)
    ttk.Button(traj_ctrl, text="Next (execute & pop)", command=on_traj_next).grid(row=0, column=4, padx=6)
    # Кнопка переключения emit для выбранного элемента
    btn_traj_toggle_emit = ttk.Button(traj_ctrl, text="Toggle Emit", command=on_traj_toggle_emit)
    btn_traj_toggle_emit.grid(row=0, column=5, padx=6)

    run_ctrl2 = ttk.Frame(traj_frame)
    run_ctrl2.pack(fill="x", padx=4, pady=2)
    ttk.Label(run_ctrl2, text="Delay (s)").grid(row=0, column=0, padx=2)
    ttk.Entry(run_ctrl2, width=6, textvariable=traj_delay_var).grid(row=0, column=1, padx=2)
    ttk.Label(run_ctrl2, text="Repeat").grid(row=0, column=2, padx=2)
    ttk.Entry(run_ctrl2, width=6, textvariable=traj_repeats_var).grid(row=0, column=3, padx=2)
    ttk.Checkbutton(run_ctrl2, text="Loop", variable=traj_loop_var).grid(row=0, column=4, padx=4)

    btn_traj_run = ttk.Button(run_ctrl2, text="Run", command=on_traj_run)
    btn_traj_run.grid(row=0, column=5, padx=6)
    btn_traj_stop = ttk.Button(run_ctrl2, text="Stop", command=on_traj_stop)
    btn_traj_stop.grid(row=0, column=6, padx=6)
    # Изначально Stop неактивна
    btn_traj_stop.configure(state="disabled")
    ttk.Button(run_ctrl2, text="Save trajectory...", command=on_traj_save).grid(row=0, column=7, padx=6)
    ttk.Button(run_ctrl2, text="Load trajectory...", command=on_traj_load).grid(row=0, column=8, padx=6)

    limits_frame = ttk.LabelFrame(ctl_frame, text="Информация о границах")
    limits_frame.pack(fill="x", padx=4, pady=4)
    ttk.Label(limits_frame, text=f"X user: {USER_LIMITS[2]['min']} .. {USER_LIMITS[2]['max']} мм (controller: {CONTROLLER_LIMITS[2]['min']}..{CONTROLLER_LIMITS[2]['max']})").pack(anchor="w", padx=4, pady=2)
    ttk.Label(limits_frame, text=f"Y user: {USER_LIMITS[1]['min']} .. {USER_LIMITS[1]['max']} мм (controller: {CONTROLLER_LIMITS[1]['min']}..{CONTROLLER_LIMITS[1]['max']})").pack(anchor="w", padx=4, pady=2)
    ttk.Label(limits_frame, text="Примечание: GUI автоматически инвертирует знак при отправке в контроллер.").pack(anchor="w", padx=4, pady=2)

    footer = ttk.Frame(ctl_frame)
    footer.pack(side="bottom", fill="x", pady=8)
    ttk.Button(footer, text="Exit", command=on_exit).pack(side="right", padx=6)

    _refresh_schedule_listbox()
    _refresh_trajectory_listbox()
    update_gui_positions()

# ----------------------------
# Главная точка входа
# ----------------------------
def main():
    global positions
    positions = load_positions_from_file()
    for m in (1, 2):
        if m not in positions:
            positions[m] = DEFAULT_USER_POSITIONS[m]
    # Попытка подключиться в фоне (если доступно)
    threading.Thread(target=connect_serial, daemon=True).start()
    start_cmd_worker()
    build_gui()
    try:
        root.mainloop()
    finally:
        save_positions_to_file()
        try:
            save_schedule_to_file(SCHEDULE_FILE)
        except Exception:
            pass
        try:
            save_trajectory_to_file(TRAJECTORY_FILE)
        except Exception:
            pass
        stop_run_schedule()
        stop_run_trajectory()
        stop_cmd_worker()
        close_serial()
        print("Завершение работы GUI.")
        
if __name__ == "__main__":
    main()
